# PiCar Real-to-Sim SUMO 강화학습 (Colab / 로컬 GPU)

실물 PiCar 코스(외곽 순환로 + 중앙 연결로 + T자 2곳)를 **미터 단위 SUMO 맵**으로 옮기고, 아래쪽 T자(J) 한 곳의 진입 허가(GO/STOP)를 DQN으로 학습합니다. 위쪽 T자(J2)는 같은 안전 제어기 + 고정 대기열 규칙으로 돕니다.
- 관측은 실차에서 얻을 수 있는 값만 씁니다 (APPROACH 표식, 정지선 IR, 출구 센서, 중앙 제어기 상태).
- `IntersectionSupervisor`만 GO를 낼 수 있고, 같은 클래스가 실차 중앙 PC(`picar_sim/real_bridge.py`)에서도 그대로 돕니다.
- 치수와 차량 값은 **가정값**입니다. 실측 후 `picar_sim/calibrate.py`로 바꾸고 재학습해야 합니다.

런타임 → 런타임 유형 변경 → GPU를 고르면 PyTorch 백엔드가 자동으로 CUDA를 씁니다. 신경망이 작아서 CPU도 충분히 빠릅니다.

결과와 가정은 `docs/PICAR_REAL2SIM_PLAN_KO.md`에 정리돼 있습니다. 처음 Colab에서 `--backend torch`를 쓸 때는 작업 공간에서 torch 경로를 실행해 보지 못했으므로, 오류가 나면 `--backend numpy`로 바꾸면 됩니다.

In [ ]:
# 1) 코드 받기: GitHub 브랜치 (푸시된 경우) 또는 ZIP 업로드
import os, subprocess
REPO = "https://github.com/leeyoungsang08/t-junction-rl-picar.git"
BRANCH = "picar-real2sim"
if not os.path.exists("t-junction-rl-picar"):
    r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO], capture_output=True, text=True)
    print(r.stdout, r.stderr)
    if r.returncode != 0:
        from google.colab import files  # 브랜치가 없으면 picar_real2sim.zip 업로드
        up = files.upload()
        subprocess.run(["unzip", "-q", "-o", next(iter(up)), "-d", "t-junction-rl-picar"], check=True)
%cd t-junction-rl-picar

In [ ]:
# 2) SUMO 설치 (pip 패키지, sudo 불필요). Colab에는 PyTorch가 이미 있습니다.
!pip -q install eclipse-sumo==1.28.0 libsumo==1.28.0 traci==1.28.0 sumolib==1.28.0 pyyaml pandas scipy matplotlib pytest
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available())

In [ ]:
# 3) 맵 생성 + 테스트 (안전 제어기, 결함 주입, SUMO 기록 -> 실차 브리지 재생 일치)
!python -m picar_sim.map_builder
!python -m pytest -q picar_sim/tests/test_picar_sim.py
from IPython.display import Image; Image("picar_sim/maps/picar_course_topview.png")

In [ ]:
# 4) 학습 3시드 + 평가. Drive에 저장하려면 OUT을 /content/drive/MyDrive/... 로 바꾸세요.
OUT = "runs/colab_cycle"
!python -m picar_sim.run_all --out {OUT} --episodes 1500 --seeds 42 43 44 --parallel 3 --backend torch

In [ ]:
# 5) 결과 보기
import pandas as pd
print(pd.read_csv(f"{OUT}/evaluation/summary_overall.csv").to_string())
print(pd.read_csv(f"{OUT}/evaluation/paired_vs_queue_rule.csv").to_string())
Image(f"{OUT}/evaluation/comparison.png")

In [ ]:
# 6) (다음 단계) 한 요소만 바꾼 실험: 직접 현시 선택 / 가정 범위 랜덤화
# !python -m picar_sim.run_all --out runs/colab_direct --control-mode direct --episodes 1500
# !python -m picar_sim.run_all --out runs/colab_rand --randomize --episodes 1500
# 끊겼으면 같은 명령에 --resume